In [1]:
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
import pandas as pd

/tmp/ipykernel_25178/2473973458.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [2]:
db = SQLDatabase.from_uri(
    "sqlite:///../data/Chinook.db"
)

In [3]:
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [4]:
table_info = db.get_table_info()

print(table_info)


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

In [6]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print(groq_api_key is not None)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

True


In [9]:
response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello! I hope you're having a wonderful day.


In [10]:
question = "Which artist has the most tracks?"

prompt = f"""
You are an expert SQLite SQL developer.

Given the following database schema:

{table_info}

Generate a valid SQLite SQL query that answers the user's question.

User Question:
{question}

Rules:
- Generate only valid SQLite SQL.
- Use only the tables and columns provided in the schema.
- Return ONLY the SQL query.
- Do not include explanations.
- Do not use markdown code blocks.

SQL:
"""

In [11]:
response = llm.invoke(prompt)

generated_sql = response.content

print(generated_sql)

SELECT a.Name
FROM Artist a
JOIN Album al ON al.ArtistId = a.ArtistId
JOIN Track t ON t.AlbumId = al.AlbumId
GROUP BY a.ArtistId
ORDER BY COUNT(t.TrackId) DESC
LIMIT 1;


In [12]:
result = db.run(generated_sql)
print(result)

[('Iron Maiden',)]


In [16]:
def generate_sql(question):
    
    prompt = f"""
    You are an expert SQLite SQL developer.

    Given the following database schema:

    {table_info}

    Generate a valid SQLite SQL query that answers the user's question.

    User Question:
    {question}

    Rules:
    - Generate only valid SQLite SQL.
    - Use only the tables and columns provided in the schema.
    - Return ONLY the SQL query.
    - Do not include explanations.
    - Do not use markdown code blocks.

    SQL:
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
question = "Which country has the highest number of customers?"

generated_sql = generate_sql(question)

print(generated_sql)

In [ ]:
result = db.run(generated_sql)

print(result)

In [ ]:
def execute_sql(sql_query):
    result = db.run(sql_query)
    return result

In [ ]:
question = "Which country has the highest number of customers?"

generated_sql = generate_sql(question)

print("Generated SQL:")
print(generated_sql)

result = execute_sql(generated_sql)

print("\nResult:")
print(result)

In [ ]:
def ask_database(question):
    sql = generate_sql(question)
    result = execute_sql(sql)

    return sql, result

In [ ]:
question = "Show the top 5 customers by total spending"

sql, result = ask_database(question)

print("Generated SQL:")
print(sql)

print("\nResult:")
print(result)

In [24]:
def validate_sql(sql_query):
    sql_query = sql_query.strip()

    # Remove one semicolon at the end
    if sql_query.endswith(";"):
        sql_query = sql_query[:-1].strip()

    # Only allow SELECT queries
    if not sql_query.upper().startswith("SELECT"):
        return False

    # Block multiple statements
    if ";" in sql_query:
        return False

    return True

In [ ]:
print(validate_sql("SELECT * FROM Customer;"))

In [ ]:
print(validate_sql("SELECT * FROM Products; DELETE FROM Customer;"))

In [29]:
def execute_sql(sql_query):

    if not validate_sql(sql_query):
        return False, "Error: Only a single SELECT query is allowed."

    try:
        result = pd.read_sql_query(sql_query, db._engine)
        return True, result

    except Exception as e:
        return False, str(e)

In [ ]:
valid_sql = """
SELECT *
FROM Customer
LIMIT 3;
"""

print(execute_sql(valid_sql))

In [ ]:
dangerous_sql = """
SELECT * FROM Customer;
DELETE FROM Customer;
"""

print(execute_sql(dangerous_sql))

In [ ]:
def ask_database(question):
    sql = generate_sql(question)
    result = execute_sql(sql)

    return sql, result

In [ ]:
question = "Show the top 5 customers by total spending"

sql, result = ask_database(question)

print("Generated SQL:")
print(sql)

print("\nResult:")
print(result)

In [22]:
def fix_sql(question, sql_query, error):

    prompt = f"""
You are an expert SQLite SQL developer.

The following SQL query produced an error.

Database schema:
{table_info}

User question:
{question}

Incorrect SQL:
{sql_query}

Database error:
{error}

Fix the SQL query.

Rules:
- Generate only valid SQLite SQL.
- Use only tables and columns from the schema.
- Return ONLY the corrected SQL.
- Do not include markdown or explanations.
"""

    response = llm.invoke(prompt)

    return response.content

In [21]:
def ask_database(question):

    # Step 1: Generate SQL
    sql = generate_sql(question)

    # Step 2: Execute SQL
    success, result = execute_sql(sql)

    # Step 3: Fix SQL if execution fails
    if not success:
        print("Initial SQL failed. Trying to fix it...")

        sql = fix_sql(
            question=question,
            sql_query=sql,
            error=result
        )

        success, result = execute_sql(sql)

    # Step 4: Generate natural language answer
    if success:
        answer = generate_answer(question, result)

        return {
            "question": question,
            "sql": sql,
            "result": result,
            "answer": answer
        }

    # If SQL still fails
    return {
        "question": question,
        "sql": sql,
        "result": None,
        "answer": f"Unable to execute the query: {result}"
    }

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)

print("Question:")
print(response["question"])

print("\nGenerated SQL:")
print(response["sql"])

print("\nResult:")
print(response["result"])

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)

print("Generated SQL:")
print(response["sql"])

print("\nResult:")
display(response["result"])

In [20]:
def generate_answer(question, result):
    
    prompt = f"""
You are a helpful data assistant.

Answer the user's question using only the SQL query result provided.

User Question:
{question}

SQL Result:
{result.to_string(index=False)}

Rules:
- Give a clear and concise answer.
- Use only information present in the SQL result.
- Do not make up information.
- Do not mention SQL unless necessary.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)
print("QUESTION:")
print(response["question"])

print("\nGENERATED SQL:")
print(response["sql"])

print("\nRESULT:")
display(response["result"])

print("\nANSWER:")
print(response["answer"])

In [30]:
questions = [
    "How many customers are there?",
    "Show all customers from Germany.",
    "Which country has the highest number of customers?",
    "Show the top 5 most expensive tracks.",
    "List all albums created by AC/DC.",
    "Which artist has the most tracks?",
    "Show the top 5 customers by total spending.",
    "Which music genre has the highest number of tracks?",
    "Which artist generated the highest revenue from track sales?",
    "Show the top 5 countries by total revenue."
]

for i, question in enumerate(questions, start=1):
    
    print("=" * 70)
    print(f"QUESTION {i}: {question}")
    print("=" * 70)

    response = ask_database(question)

    print("\nGENERATED SQL:")
    print(response["sql"])

    print("\nRESULT:")
    
    if response["result"] is not None:
        display(response["result"])

    print("\nANSWER:")
    print(response["answer"])

    print("\n")

QUESTION 1: How many customers are there?

GENERATED SQL:
SELECT COUNT(*) AS CustomerCount FROM Customer;

RESULT:


,CustomerCount
0,59



ANSWER:
There are **59** customers.


QUESTION 2: Show all customers from Germany.

GENERATED SQL:
SELECT * FROM Customer WHERE Country = 'Germany';

RESULT:


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
1,36,Hannah,Schneider,None,Tauentzienstraße 8,Berlin,None,Germany,10789,+49 030 26550280,None,hannah.schneider@yahoo.de,5
2,37,Fynn,Zimmermann,None,Berger Straße 10,Frankfurt,None,Germany,60316,+49 069 40598889,None,fzimmermann@yahoo.de,3
3,38,Niklas,Schröder,None,Barbarossastraße 19,Berlin,None,Germany,10779,+49 030 2141444,None,nschroder@surfeu.de,3



ANSWER:
**Customers from Germany**

| CustomerId | FirstName | LastName  | Company | Address                | City       | State | Country | PostalCode | Phone               | Fax | Email                         | SupportRepId |
|-----------|-----------|-----------|---------|------------------------|------------|-------|---------|------------|---------------------|-----|-------------------------------|--------------|
| 2         | Leonie    | Köhler    | None    | Theodor‑Heuss‑Straße 34| Stuttgart  | None  | Germany | 70174      | +49 0711 2842222    | None| leonekohler@surfeu.de         | 5 |
| 36        | Hannah    | Schneider | None    | Tauentzienstraße 8     | Berlin     | None  | Germany | 10789      | +49 030 26550280    | None| hannah.schneider@yahoo.de     | 5 |
| 37        | Fynn      | Zimmermann| None    | Berger Straße 10       | Frankfurt  | None  | Germany | 60316      | +49 069 40598889    | None| fzimmermann@yahoo.de          | 3 |
| 38        | Niklas    | Schröder 

...


ANSWER:
The genre with the highest number of tracks is **Rock**.


QUESTION 9: Which artist generated the highest revenue from track sales?

GENERATED SQL:
SELECT a.Name AS ArtistName,
       SUM(il.UnitPrice * il.Quantity) AS Revenue
FROM Artist a
JOIN Album al ON al.ArtistId = a.ArtistId
JOIN Track t ON t.AlbumId = al.AlbumId
JOIN InvoiceLine il ON il.TrackId = t.TrackId
GROUP BY a.ArtistId
ORDER BY Revenue DESC
LIMIT 1;

RESULT:


,ArtistName,Revenue
0,Iron Maiden,138.6



ANSWER:
The artist with the highest revenue from track sales is **Iron Maiden**, with revenue of **$138.6**.


QUESTION 10: Show the top 5 countries by total revenue.

GENERATED SQL:
SELECT BillingCountry AS Country, SUM(Total) AS TotalRevenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY TotalRevenue DESC
LIMIT 5;

RESULT:


,Country,TotalRevenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48



ANSWER:
Here are the top 5 countries by total revenue:

| Country | Total Revenue |
|---------|---------------|
| USA     | 523.06 |
| Canada  | 303.96 |
| France  | 195.10 |
| Brazil  | 190.10 |
| Germany | 156.48 |


